# Völva - Making Crystallite Size Analysis Simple

*Völva, a seeress in Norse mythology, was known for her wisdom and ability to reveal hidden truths.*

Völva is a Python-based desktop application designed for straightforward analysis of X-ray Diffraction (XRD) data to determine crystallite size and strain. It provides both Scherrer analysis and Williamson-Hall plotting capabilities, with instrumental broadening correction.

## Development

This code has been developed by Dr Nik Reeves-McLaren at the University of Sheffield, 2025.

## Features

* **Load XRD Data:** Easily load your sample and standard (for instrumental broadening correction) `.txt`, `.xy`, or `.dat` files.
* **Flexible Wavelength Selection:** Choose from common X-ray wavelengths (Cu Kα, Mo Kα, Ag Kα) or input a custom value.
* **Adjustable Constants:** Select appropriate Scherrer (K) and Williamson-Hall (K) constants.
* **Customisable Peak Finding:** Refine peak detection with parameters for prominence, minimum width, and fitting window size.
* **Scherrer Analysis:** Calculate average crystallite size from individual peak broadening, corrected for instrumental effects.
* **Williamson-Hall Analysis:** Generate a Williamson-Hall plot and perform a linear fit to separate crystallite size and strain contributions.
* **Interactive Plotting:** Visualise your XRD patterns and analysis results (detected peaks, fitted curves, WH plot) with zoom and pan functionality.
* **Plot Axis Controls:** Manually adjust the X and Y axis limits of the plot.
* **Real-time Status Log:** Monitor the application's processes, data loading, and analysis steps.

## Usage

### 1. Prerequisites

Ensure you have Python 3 installed. You will also need the following libraries. If you don't have them, install them using pip:

```bash
!pip install numpy matplotlib scipy lmfit scikit-learn
```



In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox, scrolledtext
from tkinter import ttk # Import ttk for themed widgets like Notebook
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
from scipy.signal import find_peaks, savgol_filter
from lmfit import models, Parameters
import os
import traceback # For more detailed error logging

# --- Global Constants ---
# Proximity tolerance for sample peak detection relative to standard peaks
PROXIMITY_TOLERANCE_DEG = 5.0 # degrees 2theta

class VolvaCrystalliteAnalysisApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Völva - Crystallite Size and Strain Analysis Tool")
        self.root.geometry("1400x900") # Increased window size for better layout

        self.two_theta_sample = None
        self.intensity_sample = None
        self.two_theta_std = None
        self.intensity_std = None

        # Store instrumental FWHM for each standard peak (key: 2theta, value: FWHM_deg)
        self.instrumental_fwhm = {} 
        # Store detected 2theta centers of standard peaks for filtering sample peaks
        self.std_peak_centers = [] 
        # Store info for plotting fitted standard peaks
        self.std_peaks_info = [] 

        self.lambda_nm = 0.15406 # Default for Cu Kalpha1 (nm)

        # Kalpha constants - will be used by the fitting function
        self._ka1_current = 0.15406 # Current Kalpha1 wavelength
        self._ka2_current = 0.154439 # Current Kalpha2 wavelength
        self._ka_ratio_current = 0.5 # Current Kalpha2/Kalpha1 intensity ratio
        self.enable_ka_doublet_fit = True # Flag to enable/disable doublet fitting

        # Store peak info for sample (for plotting and analysis)
        self.sample_peaks_info = [] 

        # Store plot lines for easy removal and management across subplots
        self.sample_line = None
        self.std_line = None
        self.detected_sample_peaks_scatter = None
        self.fitted_sample_curves_lines = []
        self.detected_std_peaks_scatter = None
        self.fitted_std_curves_lines = []
        self.wh_plot_line = None
        self.wh_fit_line = None

        self.create_widgets()
        self.init_plot() # Initialize plot with two subplots
        self.log_message("Völva application started. Load your XRD data.")

    def on_scherrer_k_select(self, *args):
        """Placeholder for handling Scherrer K selection changes."""
        # You can add functionality here later if needed
        self.log_message(f"Scherrer K selected: {self.scherrer_k_name_var.get()}")
        # If a plot update or analysis re-run is needed, call it here.
        # self.generate_plot()

    def create_widgets(self):
        # --- Left Control Panel ---
        self.control_panel = tk.Frame(self.root, width=400, padx=10, pady=10)
        self.control_panel.pack(side=tk.LEFT, fill=tk.Y, expand=False)
        self.control_panel.grid_rowconfigure(6, weight=1) # Allow notebook to expand

        # --- Main Plotting Area (Right) ---
        self.plot_frame = tk.Frame(self.root, padx=10, pady=10)
        self.plot_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True)

        # 1. Load Data
        self.data_frame = tk.LabelFrame(self.control_panel, text="1. Load Data", padx=10, pady=10)
        self.data_frame.grid(row=0, column=0, padx=5, pady=5, sticky="ew")

        tk.Button(self.data_frame, text="Load Sample Data", command=self.load_sample_data).pack(fill=tk.X, pady=2)
        self.sample_path_label = tk.Label(self.data_frame, text="No sample data loaded.", fg="blue", wraplength=250)
        self.sample_path_label.pack(fill=tk.X, pady=2)

        tk.Button(self.data_frame, text="Load Standard Data (LaB6)", command=self.load_std_data).pack(fill=tk.X, pady=2)
        self.std_path_label = tk.Label(self.data_frame, text="No standard data loaded.", fg="blue", wraplength=250)
        self.std_path_label.pack(fill=tk.X, pady=2)

        # 2. Parameters
        self.const_frame = tk.LabelFrame(self.control_panel, text="2. Parameters", padx=10, pady=10)
        self.const_frame.grid(row=1, column=0, padx=5, pady=5, sticky="ew")

        # Wavelength
        tk.Label(self.const_frame, text="X-ray Wavelength (nm):").grid(row=0, column=0, sticky="w", pady=2)
        self.wavelength_options = ["Cu Kα (0.15406)", "Mo Kα (0.07093)", "Ag Kα (0.05594)", "Custom"]
        self.wavelength_var = tk.StringVar(self.root)
        self.wavelength_var.set(self.wavelength_options[0])
        self.wavelength_menu = tk.OptionMenu(self.const_frame, self.wavelength_var, *self.wavelength_options, command=self.on_wavelength_select)
        self.wavelength_menu.grid(row=0, column=1, sticky="ew", padx=5)

        self.custom_wavelength_entry = tk.Entry(self.const_frame, width=15)
        self.custom_wavelength_entry.grid(row=0, column=2, padx=5)
        self.custom_wavelength_entry.insert(0, "0.15406")
        self.custom_wavelength_entry.config(state='disabled')

        # Scherrer K constant (Dropdown)
        tk.Label(self.const_frame, text="Scherrer Constant (K):").grid(row=1, column=0, sticky="w", pady="2")
        self.scherrer_k_options = [
            "General/Approximation (0.9)",
            "Spherical (FWHM) (0.94)",
            "Spherical (Integral Breadth) (0.89)",
            "Cubic (1.0)",
            "Custom"
        ]
        self.scherrer_k_var = tk.StringVar(self.root)
        self.scherrer_k_var.set(self.scherrer_k_options[0])
        self.scherrer_k_menu = tk.OptionMenu(self.const_frame, self.scherrer_k_var, *self.scherrer_k_options, command=self.on_scherrer_k_select)
        self.scherrer_k_menu.grid(row=1, column=1, sticky="ew", padx=5)

        self.custom_scherrer_k_entry = tk.Entry(self.const_frame, width=15)
        self.custom_scherrer_k_entry.grid(row=1, column=2, padx=5)
        self.custom_scherrer_k_entry.insert(0, "0.9")
        self.custom_scherrer_k_entry.config(state='disabled')

        # Williamson-Hall K constant (Entry field)
        tk.Label(self.const_frame, text="WH Constant (Kε):").grid(row=2, column=0, sticky="w", pady="2")
        self.wh_k_entry = tk.Entry(self.const_frame, width=15)
        self.wh_k_entry.grid(row=2, column=1, sticky="ew", padx=5, columnspan=2) # Span 2 columns
        self.wh_k_entry.insert(0, "4")

        # 3. Peak Finding & Fitting Parameters
        self.peak_params_frame = tk.LabelFrame(self.control_panel, text="3. Peak Finding & Fitting Parameters", padx=10, pady=10)
        self.peak_params_frame.grid(row=2, column=0, padx=5, pady=5, sticky="ew")

        tk.Label(self.peak_params_frame, text="Background Filter Window:").grid(row=0, column=0, sticky="w", pady=2)
        self.bg_filter_window_entry = tk.Entry(self.peak_params_frame, width=10)
        self.bg_filter_window_entry.grid(row=0, column=1, sticky="ew", padx=5)
        self.bg_filter_window_entry.insert(0, "151")

        tk.Label(self.peak_params_frame, text="Peak Prominence Factor (0-1):").grid(row=1, column=0, sticky="w", pady=2)
        self.peak_prominence_factor_entry = tk.Entry(self.peak_params_frame, width=10)
        self.peak_prominence_factor_entry.grid(row=1, column=1, sticky="ew", padx=5)
        self.peak_prominence_factor_entry.insert(0, "0.05")

        tk.Label(self.peak_params_frame, text="Min Peak Width (deg 2θ):").grid(row=2, column=0, sticky="w", pady=2)
        self.peak_min_width_deg_entry = tk.Entry(self.peak_params_frame, width=10)
        self.peak_min_width_deg_entry.grid(row=2, column=1, sticky="ew", padx=5)
        self.peak_min_width_deg_entry.insert(0, "0.05")

        tk.Label(self.peak_params_frame, text="Fitting Window (deg 2θ):").grid(row=3, column=0, sticky="w", pady=2)
        self.fitting_window_deg_entry = tk.Entry(self.peak_params_frame, width=10)
        self.fitting_window_deg_entry.grid(row=3, column=1, sticky="ew", padx=5)
        self.fitting_window_deg_entry.insert(0, "0.5")

        # 4. Run Analysis
        self.analysis_frame = tk.LabelFrame(self.control_panel, text="4. Run Analysis", padx=10, pady=10)
        self.analysis_frame.grid(row=3, column=0, padx=5, pady=5, sticky="ew")

        self.run_scherrer_btn = tk.Button(self.analysis_frame, text="Run Scherrer Analysis", command=self.run_scherrer_analysis)
        self.run_scherrer_btn.pack(pady=5, fill=tk.X)

        self.run_wh_btn = tk.Button(self.analysis_frame, text="Run Williamson-Hall Analysis", command=self.run_williamson_hall_analysis)
        self.run_wh_btn.pack(pady=5, fill=tk.X)

        # 5. Plot Options
        self.plot_options_frame = tk.LabelFrame(self.control_panel, text="5. Plot Options", padx=10, pady=10)
        self.plot_options_frame.grid(row=4, column=0, padx=5, pady=5, sticky="ew")

        self.show_sample_peaks_var = tk.BooleanVar(value=True)
        tk.Checkbutton(self.plot_options_frame, text="Show Sample Peaks", variable=self.show_sample_peaks_var, command=self.update_plot).pack(anchor=tk.W)

        self.show_sample_fits_var = tk.BooleanVar(value=True)
        tk.Checkbutton(self.plot_options_frame, text="Show Sample Fits", variable=self.show_sample_fits_var, command=self.update_plot).pack(anchor=tk.W)

        self.show_std_peaks_var = tk.BooleanVar(value=True)
        tk.Checkbutton(self.plot_options_frame, text="Show Standard Peaks", variable=self.show_std_peaks_var, command=self.update_plot).pack(anchor=tk.W)

        self.show_std_fits_var = tk.BooleanVar(value=True)
        tk.Checkbutton(self.plot_options_frame, text="Show Standard Fits", variable=self.show_std_fits_var, command=self.update_plot).pack(anchor=tk.W)


        # Notebook for Results and Log
        self.notebook = ttk.Notebook(self.control_panel)
        self.notebook.grid(row=5, column=0, padx=5, pady=5, sticky="nsew", rowspan=2) # Spanning more rows, weight=1 on row 5/6
        self.control_panel.grid_rowconfigure(5, weight=1) # Make this row expandable

        # Scherrer Results Tab
        self.scherrer_tab = ttk.Frame(self.notebook)
        self.notebook.add(self.scherrer_tab, text="Scherrer Results")
        self.scherrer_results_text_widget = scrolledtext.ScrolledText(self.scherrer_tab, wrap=tk.WORD)
        self.scherrer_results_text_widget.pack(expand=True, fill="both", padx=5, pady=5)
        self.scherrer_results_text_widget.config(state='disabled')

        # Williamson-Hall Results Tab
        self.wh_tab = ttk.Frame(self.notebook)
        self.notebook.add(self.wh_tab, text="WH Results")
        self.wh_results_text_widget = scrolledtext.ScrolledText(self.wh_tab, wrap=tk.WORD)
        self.wh_results_text_widget.pack(expand=True, fill="both", padx=5, pady=5)
        self.wh_results_text_widget.config(state='disabled')

        # Activity Log Tab
        self.log_tab = ttk.Frame(self.notebook)
        self.notebook.add(self.log_tab, text="Activity Log")
        self.log_text_widget = scrolledtext.ScrolledText(self.log_tab, wrap=tk.WORD)
        self.log_text_widget.pack(expand=True, fill="both", padx=5, pady=5)
        self.log_text_widget.config(state='disabled')


    def init_plot(self):
        # Create a figure with two subplots, sharing the x-axis
        self.fig, (self.ax_sample, self.ax_std) = plt.subplots(nrows=2, ncols=1, figsize=(10, 10), sharex=True)
        self.canvas = FigureCanvasTkAgg(self.fig, master=self.plot_frame)
        self.canvas_widget = self.canvas.get_tk_widget()
        self.canvas_widget.pack(side=tk.TOP, fill=tk.BOTH, expand=1)

        self.toolbar = NavigationToolbar2Tk(self.canvas, self.plot_frame)
        self.toolbar.update()
        self.canvas_widget.pack(side=tk.TOP, fill=tk.BOTH, expand=1)

        # Set up sample subplot
        self.ax_sample.set_ylabel('Intensity (Sample)')
        self.ax_sample.set_title('XRD Patterns')
        self.ax_sample.grid(True)
        self.sample_line, = self.ax_sample.plot([], [], label='Sample Data', color='blue')
        self.detected_sample_peaks_scatter = self.ax_sample.scatter([], [], color='red', marker='x', label='Detected Peaks')
        # Fitted curves will be added dynamically to self.fitted_sample_curves_lines

        # Set up standard subplot
        self.ax_std.set_xlabel(r'$2\theta$ / $^\circ$')
        self.ax_std.set_ylabel('Intensity (Standard)')
        self.ax_std.grid(True)
        self.std_line, = self.ax_std.plot([], [], label='Standard Data (LaB6)', color='green', linestyle=':')
        self.detected_std_peaks_scatter = self.ax_std.scatter([], [], color='purple', marker='o', label='Detected Peaks')
        # Fitted curves will be added dynamically to self.fitted_std_curves_lines

        self.fig.tight_layout(rect=[0, 0.05, 1, 0.98]) # Adjust layout
        self.canvas.draw()

    def _set_results_text(self, widget, text):
        widget.config(state='normal')
        widget.delete(1.0, tk.END)
        widget.insert(tk.END, text)
        widget.config(state='disabled')
        widget.see(tk.END)

    def log_message(self, message):
        self._set_results_text(self.log_text_widget, self.log_text_widget.get(1.0, tk.END).strip() + "\n" + message)

    def on_wavelength_select(self, *args):
        selected_option = self.wavelength_var.get()
        if "Cu Kα" in selected_option:
            self.custom_wavelength_entry.config(state='disabled')
            self.custom_wavelength_entry.delete(0, tk.END)
            self.custom_wavelength_entry.insert(0, "0.15406") # Update display
            self._ka1_current = 0.15406
            self._ka2_current = 0.154439
            self._ka_ratio_current = 0.5
            self.enable_ka_doublet_fit = True
            self.log_message("Copper Kα selected. Kα1-Kα2 deconvolution enabled.")
        elif "Mo Kα" in selected_option:
            self.custom_wavelength_entry.config(state='disabled')
            self.custom_wavelength_entry.delete(0, tk.END)
            self.custom_wavelength_entry.insert(0, "0.07093") # Update display
            self._ka1_current = 0.07093
            self._ka2_current = 0.071073 # Molybdenum Kalpha2
            self._ka_ratio_current = 0.5 # Approx ratio for Mo
            self.enable_ka_doublet_fit = True
            self.log_message("Molybdenum Kα selected. Kα1-Kα2 deconvolution enabled.")
        elif "Ag Kα" in selected_option:
            self.custom_wavelength_entry.config(state='disabled')
            self.custom_wavelength_entry.delete(0, tk.END)
            self.custom_wavelength_entry.insert(0, "0.05594") # Update display
            self._ka1_current = 0.05594
            self._ka2_current = 0.05638 # Silver Kalpha2
            self._ka_ratio_current = 0.5 # Approx ratio for Ag
            self.enable_ka_doublet_fit = True
            self.log_message("Silver Kα selected. Kα1-Kα2 deconvolution enabled.")
        elif selected_option == "Custom":
            self.custom_wavelength_entry.config(state='normal')
            self.enable_ka_doublet_fit = False
            self.log_message("Custom wavelength selected. Kα1-Kα2 deconvolution is disabled.")

        self.get_wavelength() # This sets self.lambda_nm from custom_wavelength_entry or selected_option


    def get_wavelength(self):
        try:
            if self.wavelength_var.get() == "Custom":
                val = float(self.custom_wavelength_entry.get())
                self._ka1_current = val # For custom, assume single value is Kalpha1
            else: # For predefined options, self._ka1_current is already set by on_wavelength_select
                val = self._ka1_current # Use the already set Kalpha1

            if val <= 0:
                messagebox.showwarning("Input Error", "Wavelength must be a positive number.")
                self.log_message("Error: Invalid wavelength value (non-positive).")
                return None
            self.lambda_nm = val # This is always the Kalpha1 wavelength used for Scherrer/WH formulas
            return self.lambda_nm
        except ValueError:
            messagebox.showerror("Input Error", "Invalid wavelength value. Please enter a number.")
            self.log_message("Error: Invalid wavelength value (not a number).")
            return None


    def get_k_scherrer(self):
        try:
            if self.scherrer_k_var.get() == "Custom":
                val = float(self.custom_scherrer_k_entry.get())
            else:
                # Extract number from string like "General/Approximation (0.9)"
                val_str = self.scherrer_k_var.get().split('(')[1].replace(')', '')
                val = float(val_str)
            if val <= 0:
                messagebox.showwarning("Input Error", "Scherrer K constant must be a positive number.")
                self.log_message("Error: Invalid Scherrer K constant (non-positive).")
                return None
            return val
        except ValueError:
            messagebox.showerror("Input Error", "Invalid Scherrer K constant. Please enter a number or select from options.")
            self.log_message("Error: Invalid Scherrer K constant (not a number).")
            return None


    def get_k_wh(self):
        try:
            val = float(self.wh_k_entry.get())
            if val <= 0:
                messagebox.showwarning("Input Error", "Williamson-Hall K constant must be a positive number.")
                self.log_message("Error: Invalid Williamson-Hall K constant (non-positive).")
                return None
            return val
        except ValueError:
            messagebox.showerror("Input Error", "Invalid Williamson-Hall K constant. Please enter a number.")
            self.log_message("Error: Invalid Williamson-Hall K constant (not a number).")
            return None

    def get_bg_filter_window(self):
        try:
            window = int(self.bg_filter_window_entry.get())
            if window % 2 == 0:
                messagebox.showwarning("Input Warning", "Background filter window should be an odd number. Adjusting to nearest odd.")
                window += 1 # Make it odd if even
            if window <= 0:
                messagebox.showwarning("Input Error", "Background filter window must be positive.")
                self.log_message("Error: Background filter window must be positive.")
                return None
            return window
        except ValueError:
            messagebox.showerror("Input Error", "Invalid background filter window. Please enter an integer.")
            self.log_message("Error: Invalid background filter window (not an integer).")
            return None

    def get_peak_prominence_factor(self):
        try:
            factor = float(self.peak_prominence_factor_entry.get())
            if not (0 <= factor <= 1):
                messagebox.showwarning("Input Warning", "Peak prominence factor should be between 0 and 1.")
            return factor
        except ValueError:
            messagebox.showerror("Input Error", "Invalid peak prominence factor. Please enter a number.")
            self.log_message("Error: Invalid peak prominence factor (not a number).")
            return None

    def get_peak_min_width_deg(self):
        try:
            val = float(self.peak_min_width_deg_entry.get())
            if val <= 0:
                messagebox.showwarning("Input Error", "Minimum peak width must be a positive number.")
                self.log_message("Error: Minimum peak width must be positive.")
                return None
            return val
        except ValueError:
            messagebox.showerror("Input Error", "Invalid minimum peak width. Please enter a number.")
            self.log_message("Error: Invalid minimum peak width (not a number).")
            return None

    def get_fitting_window_deg(self):
        try:
            val = float(self.fitting_window_deg_entry.get())
            if val <= 0:
                messagebox.showwarning("Input Error", "Fitting window must be a positive number.")
                self.log_message("Error: Fitting window must be positive.")
                return None
            return val
        except ValueError:
            messagebox.showerror("Input Error", "Invalid fitting window. Please enter a number.")
            self.log_message("Error: Invalid fitting window (not a number).")
            return None

    def load_data_file(self):
        file_path = filedialog.askopenfilename(
            filetypes=[("XRD Data Files", "*.txt *.xy *.dat"), ("All Files", "*.*")]
        )
        if not file_path:
            return None, None, None

        try:
            data = np.loadtxt(file_path)
            two_theta = data[:, 0]
            intensity = data[:, 1]
            return two_theta, intensity, file_path
        except Exception as e:
            messagebox.showerror("File Load Error", f"Could not load data from {os.path.basename(file_path)}.\nError: {e}")
            self.log_message(f"Error loading data from {os.path.basename(file_path)}: {e}")
            return None, None, None

    def load_sample_data(self):
        self.two_theta_sample, self.intensity_sample, file_path = self.load_data_file()
        if self.two_theta_sample is not None:
            self.sample_path_label.config(text=f"Sample: {os.path.basename(file_path)}")
            self.log_message(f"Sample data loaded: {os.path.basename(file_path)}")
            self.update_plot()
            if self.two_theta_std is not None:
                self.log_message("Both sample and standard loaded. Ready for analysis.")
            else:
                self.log_message("Sample data loaded. Please load standard data.")

    def load_std_data(self):
        self.two_theta_std, self.intensity_std, file_path = self.load_data_file()
        if self.two_theta_std is not None:
            self.std_path_label.config(text=f"Standard: {os.path.basename(file_path)}")
            self.log_message(f"Standard data loaded: {os.path.basename(file_path)}")
            self.log_message("Calculating instrumental broadening from standard data...")
            self.calculate_instrumental_broadening()
            self.update_plot() # Update plot to show standard data and its fits
            if self.two_theta_sample is not None:
                self.log_message("Both sample and standard loaded. Ready for analysis.")

    def update_plot(self):
        # Clear all previous plot elements
        self.ax_sample.clear()
        self.ax_std.clear()

        # Reset titles and labels for standard view
        self.ax_sample.set_title('XRD Patterns')
        self.ax_sample.set_ylabel('Intensity (Sample)')
        self.ax_std.set_xlabel(r'$2\theta$ / $^\circ$')
        self.ax_std.set_ylabel('Intensity (Standard)')

        # Plot Sample Data
        if self.two_theta_sample is not None and self.intensity_sample is not None:
            self.ax_sample.plot(self.two_theta_sample, self.intensity_sample, label='Sample Data', color='blue')
            if self.show_sample_peaks_var.get() and hasattr(self, 'sample_peaks_info'):
                peak_2thetas = [info['2theta'] for info in self.sample_peaks_info]
                peak_intensities = [info['peak_intensity'] for info in self.sample_peaks_info]
                self.ax_sample.scatter(peak_2thetas, peak_intensities, color='red', marker='x', label='Detected Peaks')
            if self.show_sample_fits_var.get() and hasattr(self, 'sample_peaks_info'):
                for peak_info in self.sample_peaks_info:
                    if 'fit_x' in peak_info and 'fit_y' in peak_info:
                        self.ax_sample.plot(peak_info['fit_x'], peak_info['fit_y'], color='orange', linestyle='--', alpha=0.7)

        # Plot Standard Data
        if self.two_theta_std is not None and self.intensity_std is not None:
            self.ax_std.plot(self.two_theta_std, self.intensity_std, label='Standard Data (LaB6)', color='green', linestyle=':')
            if self.show_std_peaks_var.get() and hasattr(self, 'std_peaks_info'):
                peak_2thetas = [info['2theta'] for info in self.std_peaks_info]
                peak_intensities = [info['peak_intensity'] for info in self.std_peaks_info]
                self.ax_std.scatter(peak_2thetas, peak_intensities, color='purple', marker='o', label='Detected Peaks')
            if self.show_std_fits_var.get() and hasattr(self, 'std_peaks_info'):
                for peak_info in self.std_peaks_info:
                    if 'fit_x' in peak_info and 'fit_y' in peak_info:
                        self.ax_std.plot(peak_info['fit_x'], peak_info['fit_y'], color='cyan', linestyle='--', alpha=0.7)

        # Add legends and grids
        self.ax_sample.legend(loc='best', fontsize=8)
        self.ax_sample.grid(True)
        self.ax_std.legend(loc='best', fontsize=8)
        self.ax_std.grid(True)

        self.fig.tight_layout(rect=[0, 0.05, 1, 0.98])
        self.canvas.draw_idle()


    def _calculate_delta_2theta(self, theta_rad, lambda_ka1, lambda_ka2):
        """Calculate the theoretical 2theta separation between Kalpha1 and Kalpha2."""
        if lambda_ka1 == 0 or np.isclose(theta_rad, 0) or np.tan(theta_rad) == 0: # Avoid division by zero or tan(0)
            return 0
        delta_lambda = lambda_ka2 - lambda_ka1
        # Common formula uses tan(theta) * (delta_lambda / lambda_ka1) for delta(theta)
        # So for delta(2theta) it's 2 * delta(theta)
        return 2 * np.rad2deg(np.tan(theta_rad) * (delta_lambda / lambda_ka1))

    def _pseudo_voigt_fwhm(self, sigma, fraction):
        """
        Calculates the FWHM of a Pseudo-Voigt function.
        sigma: Gaussian standard deviation from lmfit fit.
        fraction: Lorentzian character from lmfit fit.
        """
        # FWHM_Gaussian = 2 * sigma * sqrt(2 * ln(2))
        fwhm_g = 2 * sigma * np.sqrt(2 * np.log(2))
        # FWHM_Lorentzian = 2 * sigma (if gamma=sigma for Lorentzian)
        # However, lmfit's PseudoVoigtModel uses sigma directly, and gamma is derived from sigma
        # The FWHM of a Lorentzian with sigma parameter (HWHM) is sigma. So FWHM is 2*sigma.
        fwhm_l = 2 * sigma # For lmfit's PseudoVoigt, sigma is related to HWHM directly for the Lorentzian part

        # Combined FWHM for Pseudo-Voigt
        return (fwhm_g * (1 - fraction)) + (fwhm_l * fraction)


    def _fit_xrd_peak(self, x_data, y_data, peak_2theta_guess, initial_sigma_guess, fitting_window_deg):
        """
        Fits an XRD peak, optionally as a Kalpha1-Kalpha2 doublet based on self.enable_ka_doublet_fit.

        Returns:
            dict: containing 'success', 'center', 'fwhm', 'amplitude', 'fit_x', 'fit_y', 'message'.
                  'center', 'fwhm', 'amplitude' refer to Kalpha1 component if doublet fit.
        """
        search_min = peak_2theta_guess - fitting_window_deg / 2
        search_max = peak_2theta_guess + fitting_window_deg / 2

        mask = (x_data >= search_min) & (x_data <= search_max)
        x_range = x_data[mask]
        y_range = y_data[mask]

        if len(x_range) < 10:
            return {'success': False, 'message': f"Insufficient data points ({len(x_range)}) in fitting window."}

        amplitude_guess = np.max(y_range)
        center_guess = peak_2theta_guess
        sigma_guess = max(initial_sigma_guess, (x_range.max() - x_range.min()) / 5)
        if sigma_guess == 0: sigma_guess = 0.01

        fit_result = None

        if self.enable_ka_doublet_fit:
            # Calculate theoretical Kalpha1-Kalpha2 separation for the guess position
            theta_ka1_guess_rad = np.deg2rad(center_guess / 2.0)
            delta_2theta_ka1_ka2 = self._calculate_delta_2theta(theta_ka1_guess_rad, self._ka1_current, self._ka2_current)

            # Kalpha1 model
            model1 = models.PseudoVoigtModel(prefix='k1_')
            params1 = model1.make_params(amplitude=amplitude_guess, center=center_guess, sigma=sigma_guess, fraction=0.5)

            # Kalpha2 model
            model2 = models.PseudoVoigtModel(prefix='k2_')
            # Combine models
            composite_model = model1 + model2
            params = params1 # Start with k1 params

            # Add Kalpha2 parameters with expressions
            params.add('k2_center', expr=f'k1_center + {delta_2theta_ka1_ka2}')
            params.add('k2_amplitude', expr=f'k1_amplitude * {self._ka_ratio_current}')
            params.add('k2_sigma', expr='k1_sigma')
            params.add('k2_fraction', expr='k1_fraction') # Same peak shape

            # Set bounds for Kalpha1 (main peak)
            params['k1_center'].min = search_min
            params['k1_center'].max = search_max
            params['k1_sigma'].min = 0.001
            params['k1_sigma'].max = (x_range.max() - x_range.min()) / 2
            params['k1_amplitude'].min = 0

            try:
                fit_result = composite_model.fit(y_range, params, x=x_range)
            except Exception as e:
                return {'success': False, 'message': f"lmfit error during doublet fit: {e}"}


            if fit_result.success:
                # Retrieve individual components' parameters after the fit
                k1_sigma = fit_result.best_values['k1_sigma']
                k1_fraction = fit_result.best_values['k1_fraction']
                # Manually calculate FWHM for the Kalpha1 component
                fwhm_extracted = self._pseudo_voigt_fwhm(k1_sigma, k1_fraction)

                center_extracted = fit_result.best_values['k1_center'] # Center of Kalpha1
                amplitude_extracted = fit_result.best_values['k1_amplitude'] # Amplitude of Kalpha1
            else:
                return {'success': False, 'message': fit_result.error_message}

        else: # Single PseudoVoigt fit
            model = models.PseudoVoigtModel()
            params = model.make_params(amplitude=amplitude_guess, center=center_guess, sigma=sigma_guess, fraction=0.5)
            params['center'].min = search_min
            params['center'].max = search_max
            params['sigma'].min = 0.001
            params['sigma'].max = (x_range.max() - x_range.min()) / 2
            params['amplitude'].min = 0

            try:
                fit_result = model.fit(y_range, params, x=x_range)
            except Exception as e:
                return {'success': False, 'message': f"lmfit error during single fit: {e}"}

            if fit_result.success:
                fwhm_extracted = fit_result.best_values['fwhm']
                center_extracted = fit_result.best_values['center']
                amplitude_extracted = fit_result.best_values['amplitude']
            else:
                return {'success': False, 'message': fit_result.error_message}

        # Evaluate fit for plotting
        fit_x = np.linspace(x_range.min(), x_range.max(), 500)
        fit_y = fit_result.eval(x=fit_x)


        return {
            'success': True,
            'center': center_extracted,
            'fwhm': fwhm_extracted,
            'amplitude': amplitude_extracted,
            'fit_x': fit_x,
            'fit_y': fit_y,
            'message': 'Fit successful.'
        }


    def calculate_instrumental_broadening(self):
        if self.two_theta_std is None or self.intensity_std is None:
            self.log_message("Warning: Cannot calculate instrumental broadening without standard data.")
            return

        bg_window = self.get_bg_filter_window()
        prom_factor = self.get_peak_prominence_factor()
        min_width = self.get_peak_min_width_deg()
        fitting_window = self.get_fitting_window_deg()

        if any(p is None for p in [bg_window, prom_factor, min_width, fitting_window]):
            self.log_message("Error: Invalid peak finding/fitting parameters for standard data.")
            return

        self.instrumental_fwhm = {}
        self.std_peak_centers = []
        self.std_peaks_info = [] # Reset for plotting

        self.log_message("Starting instrumental FWHM calculation...")

        # Background subtract standard data
        bg_std = savgol_filter(self.intensity_std, bg_window, 3)
        processed_std_intensity = self.intensity_std - bg_std
        processed_std_intensity[processed_std_intensity < 0] = 0

        # Find peaks in standard
        step_size = self.two_theta_std[1] - self.two_theta_std[0]
        min_width_points = min_width / step_size if step_size > 0 else 1 

        std_peaks_indices, _ = find_peaks(processed_std_intensity,
                                         height=np.max(processed_std_intensity) * prom_factor,
                                         width=min_width_points)

        if not std_peaks_indices.size:
            self.log_message("No significant peaks found in standard data for instrumental broadening calculation. Adjust peak finding parameters.")
            messagebox.showwarning("No Peaks", "No significant peaks found in standard data. Adjust peak finding parameters.")
            return

        for idx in std_peaks_indices:
            peak_2theta_guess = self.two_theta_std[idx]

            # Use a rough sigma guess for the _fit_xrd_peak
            x_diff = self.two_theta_std[-1] - self.two_theta_std[0]
            if x_diff == 0: x_diff = 1 # Avoid division by zero
            rough_sigma_guess = (min_width / 2.355) if min_width > 0 else (x_diff / len(self.two_theta_std) * 5) # Default to a reasonable value
            if rough_sigma_guess == 0: rough_sigma_guess = 0.01

            fit_result_dict = self._fit_xrd_peak(
                x_data=self.two_theta_std,
                y_data=processed_std_intensity,
                peak_2theta_guess=peak_2theta_guess,
                initial_sigma_guess=rough_sigma_guess,
                fitting_window_deg=fitting_window
            )

            if not fit_result_dict['success']:
                self.log_message(f"  Fit failed for standard peak at {peak_2theta_guess:.2f}°: {fit_result_dict['message']}")
                continue 

            fwhm_instrumental = fit_result_dict['fwhm']
            fitted_center = fit_result_dict['center']

            self.instrumental_fwhm[fitted_center] = fwhm_instrumental
            self.std_peak_centers.append(fitted_center) # Store center for sample filtering
            self.log_message(f"  Standard peak at {fitted_center:.2f}°: FWHM_inst = {fwhm_instrumental:.4f}° (Fit success: True)")

            # Store info for plotting fitted standard peaks
            self.std_peaks_info.append({
                '2theta': fitted_center,
                'peak_intensity': fit_result_dict['amplitude'],
                'fit_x': fit_result_dict['fit_x'],
                'fit_y': fit_result_dict['fit_y']
            })

        if not self.instrumental_fwhm:
            messagebox.showwarning("Calculation Error", "Failed to calculate instrumental broadening for any standard peak. Check standard data or parameters.")
            self.log_message("Error: No instrumental FWHM calculated.")
        else:
            self.log_message("Instrumental broadening calculation complete.")
            self.log_message("Instrumental FWHM values calculated:")
            for center, fwhm in self.instrumental_fwhm.items():
                self.log_message(f"  2θ: {center:.2f}°, FWHM_inst: {fwhm:.4f}°")
        self.update_plot() # Update plot to show standard data and its fits


    def _process_sample_peaks_for_analysis(self):
        """
        Helper function to find, fit, and correct sample peaks.
        Filters sample peaks based on proximity to standard peaks.
        Returns a list of dictionaries with analysis results for each peak.
        """
        if self.two_theta_sample is None or self.intensity_sample is None:
            self.log_message("Warning: No sample data loaded for processing.")
            return []

        wavelength_nm = self.get_wavelength()

        bg_window = self.get_bg_filter_window()
        prom_factor = self.get_peak_prominence_factor()
        min_width = self.get_peak_min_width_deg()
        fitting_window = self.get_fitting_window_deg()

        if any(p is None for p in [wavelength_nm, bg_window, prom_factor, min_width, fitting_window]):
            self.log_message("Error: Invalid parameters for sample peak processing.")
            return []

        # Background subtract sample data
        bg_sample = savgol_filter(self.intensity_sample, bg_window, 3)
        processed_sample_intensity = self.intensity_sample - bg_sample
        processed_sample_intensity[processed_sample_intensity < 0] = 0 

        # Find all raw peaks in sample
        step_size = self.two_theta_sample[1] - self.two_theta_sample[0]
        min_width_points = min_width / step_size if step_size > 0 else 1

        raw_sample_peaks_indices, _ = find_peaks(processed_sample_intensity,
                                            height=np.max(processed_sample_intensity) * prom_factor,
                                            width=min_width_points)

        if not raw_sample_peaks_indices.size:
            self.log_message("No significant peaks found in sample data with current parameters.")
            return []

        # Filter sample peaks based on proximity to standard peaks
        filtered_sample_peaks_indices = []
        if self.std_peak_centers:
            self.log_message(f"Filtering sample peaks within {PROXIMITY_TOLERANCE_DEG}° of standard peaks...")
            for s_idx in raw_sample_peaks_indices:
                sample_peak_2theta = self.two_theta_sample[s_idx]
                is_near_std_peak = False
                for std_center in self.std_peak_centers:
                    if abs(sample_peak_2theta - std_center) <= PROXIMITY_TOLERANCE_DEG:
                        is_near_std_peak = True
                        break
                if is_near_std_peak:
                    filtered_sample_peaks_indices.append(s_idx)
            self.log_message(f"Found {len(filtered_sample_peaks_indices)} sample peaks near standard peaks.")
        else:
            self.log_message("No standard peak centers available for filtering sample peaks. Analyzing all detected peaks.")
            filtered_sample_peaks_indices = raw_sample_peaks_indices # Use all detected if no standard centers

        if not filtered_sample_peaks_indices:
            self.log_message("No sample peaks found after filtering by standard peak proximity.")
            return []

        self.sample_peaks_info = [] # Reset for plotting
        analysis_results = []

        self.log_message(f"Fitting {len(filtered_sample_peaks_indices)} sample peaks...")

        for idx in filtered_sample_peaks_indices:
            peak_2theta_obs_guess = self.two_theta_sample[idx]

            # Use a rough sigma guess for the _fit_xrd_peak
            x_diff = self.two_theta_sample[-1] - self.two_theta_sample[0]
            if x_diff == 0: x_diff = 1 # Avoid division by zero
            rough_sigma_guess = (min_width / 2.355) if min_width > 0 else (x_diff / len(self.two_theta_sample) * 5) # Default to a reasonable value
            if rough_sigma_guess == 0: rough_sigma_guess = 0.01

            fit_result_dict = self._fit_xrd_peak(
                x_data=self.two_theta_sample,
                y_data=processed_sample_intensity,
                peak_2theta_guess=peak_2theta_obs_guess,
                initial_sigma_guess=rough_sigma_guess,
                fitting_window_deg=fitting_window
            )

            if not fit_result_dict['success']:
                self.log_message(f"  Fit failed for sample peak at {peak_2theta_obs_guess:.2f}°: {fit_result_dict['message']}")
                continue

            fwhm_obs = fit_result_dict['fwhm']
            peak_center = fit_result_dict['center']
            peak_intensity = fit_result_dict['amplitude']

            fwhm_inst = 0.0
            if self.instrumental_fwhm:
                closest_std_peak_2theta = min(self.instrumental_fwhm.keys(), key=lambda k: abs(k - peak_center))
                fwhm_inst = self.instrumental_fwhm[closest_std_peak_2theta]
                self.log_message(f"  Sample peak at {peak_center:.2f} deg 2theta: Observed FWHM = {fwhm_obs:.4f}°, Instrumental FWHM (closest standard): {fwhm_inst:.4f}°")
            else:
                self.log_message(f"  Sample peak at {peak_center:.2f} deg 2theta: Observed FWHM = {fwhm_obs:.4f}° (No instrumental correction applied).")


            # Corrected FWHM (Beta_hkl for size/strain broadening)
            if fwhm_obs > fwhm_inst:
                beta_corrected_deg = np.sqrt(fwhm_obs**2 - fwhm_inst**2)
            else:
                beta_corrected_deg = 0.0 

            theta_rad = np.deg2rad(peak_center / 2.0)

            analysis_results.append({
                '2theta': peak_center,
                'FWHM_obs': fwhm_obs,
                'FWHM_inst': fwhm_inst,
                'Beta_corrected_deg': beta_corrected_deg, # Store in degrees for consistency before rad conversion
                'theta_rad': theta_rad
            })

            # Store info for plotting fitted curves
            self.sample_peaks_info.append({
                '2theta': peak_center,
                'peak_intensity': peak_intensity,
                'fit_x': fit_result_dict['fit_x'],
                'fit_y': fit_result_dict['fit_y']
            })

        self.log_message("Sample peak processing complete.")
        self.update_plot() # Update plot to show detected peaks and fitted curves
        return analysis_results


    def run_scherrer_analysis(self):
        self._set_results_text(self.scherrer_results_text_widget, "Running Scherrer analysis...")
        self.notebook.select(self.scherrer_tab)

        if self.two_theta_sample is None:
            messagebox.showwarning("Missing Data", "Please load sample data first.")
            self.log_message("Warning: Cannot run Scherrer analysis without sample data.")
            return

        if not self.instrumental_fwhm:
            resp = messagebox.askyesno("Missing Instrumental Broadening", 
                                        "Instrumental broadening not calculated. Proceed without correction? (Not Recommended)")
            if not resp:
                self.log_message("Scherrer analysis aborted due to missing instrumental FWHM.")
                self._set_results_text(self.scherrer_results_text_widget, "Analysis aborted.")
                return
            self.log_message("Warning: Proceeding with Scherrer analysis WITHOUT instrumental broadening correction.")

        k_scherrer = self.get_k_scherrer()
        if k_scherrer is None: return

        analysis_results = self._process_sample_peaks_for_analysis()
        if not analysis_results:
            self._set_results_text(self.scherrer_results_text_widget, "No peaks successfully processed for Scherrer method.")
            self.log_message("No peaks successfully processed for Scherrer method.")
            return

        scherrer_sizes = []
        results_str_lines = ["Scherrer Analysis Results:"]

        for res in analysis_results:
            beta_corrected_rad = np.deg2rad(res['Beta_corrected_deg'])
            cos_theta = np.cos(res['theta_rad'])

            if beta_corrected_rad > 0 and cos_theta > 0:
                crystallite_size_nm = (k_scherrer * self.lambda_nm) / (beta_corrected_rad * cos_theta)
            else:
                crystallite_size_nm = np.nan 

            scherrer_sizes.append(crystallite_size_nm)

            size_str = f"{crystallite_size_nm:.2f} nm" if not np.isnan(crystallite_size_nm) else "N/A"
            results_str_lines.append(f"Peak at {res['2theta']:.2f} deg 2θ:")
            results_str_lines.append(f"  FWHM (Observed): {res['FWHM_obs']:.4f}°")
            results_str_lines.append(f"  FWHM (Instrumental): {res['FWHM_inst']:.4f}°")
            results_str_lines.append(f"  Beta (Corrected, Degrees): {res['Beta_corrected_deg']:.4f}°")
            results_str_lines.append(f"  Crystallite Size (D): {size_str}")
            results_str_lines.append("-" * 30)

        valid_scherrer_sizes = [s for s in scherrer_sizes if not np.isnan(s)]
        if valid_scherrer_sizes:
            avg_size = np.nanmean(valid_scherrer_sizes)
            std_dev_size = np.nanstd(valid_scherrer_sizes)
            results_str_lines.append(f"\nAverage Crystallite Size: {avg_size:.1f} nm (Std Dev: {std_dev_size:.1f} nm)")
        else:
            results_str_lines.append("\nNo valid crystallite sizes could be calculated.")

        self._set_results_text(self.scherrer_results_text_widget, "\n".join(results_str_lines))
        self.log_message("Scherrer analysis complete.")


    def run_williamson_hall_analysis(self):
        self._set_results_text(self.wh_results_text_widget, "Running Williamson-Hall analysis...")
        self.notebook.select(self.wh_tab)

        if self.two_theta_sample is None:
            messagebox.showwarning("Missing Data", "Please load sample data first.")
            self.log_message("Warning: Cannot perform Williamson-Hall analysis without sample data.")
            return

        if not self.instrumental_fwhm:
            resp = messagebox.askyesno("Missing Instrumental Broadening", 
                                        "Instrumental broadening not calculated. Proceed without correction? (Not Recommended)")
            if not resp:
                self.log_message("Williamson-Hall analysis aborted due to missing instrumental FWHM.")
                self._set_results_text(self.wh_results_text_widget, "Analysis aborted.")
                return
            self.log_message("Warning: Proceeding with Williamson-Hall analysis WITHOUT instrumental broadening correction.")

        k_wh = self.get_k_wh() # Use WH K constant
        if k_wh is None: return

        analysis_results = self._process_sample_peaks_for_analysis()
        if not analysis_results:
            self._set_results_text(self.wh_results_text_widget, "No peaks successfully processed for Williamson-Hall method.")
            self.log_message("No peaks successfully processed for Williamson-Hall method.")
            return

        # Prepare data for Williamson-Hall plot
        wh_x = [] # 4 * sin(theta)
        wh_y = [] # Beta_hkl * cos(theta)

        for res in analysis_results:
            beta_hkl_rad = np.deg2rad(res['Beta_corrected_deg'])
            theta_rad = res['theta_rad']

            if beta_hkl_rad > 0 and theta_rad > 0:
                wh_x.append(4 * np.sin(theta_rad))
                wh_y.append(beta_hkl_rad * np.cos(theta_rad))

        if not wh_x or len(wh_x) < 2:
            self._set_results_text(self.wh_results_text_widget, "Not enough valid peaks (at least 2) for Williamson-Hall analysis after correction.")
            self.log_message("Error: Not enough valid peaks (at least 2) for Williamson-Hall analysis after correction.")
            self.update_plot() # Revert plot to standard view
            return

        self.wh_x = np.array(wh_x)
        self.wh_y = np.array(wh_y)

        try:
            # Linear regression: y = slope * x + intercept
            self.wh_slope, self.wh_intercept = np.polyfit(self.wh_x, self.wh_y, 1)

            # Calculate Crystallite Size and Strain
            # Intercept = K * lambda / D_WH -> D_WH = K * lambda / Intercept
            # Slope = epsilon (strain)

            size_wh_nm = np.nan
            if self.wh_intercept > 0: 
                size_wh_nm = (k_wh * self.lambda_nm) / self.wh_intercept

            strain_wh = self.wh_slope / 4.0 # Strain is dimensionless (epsilon)

            results_lines = [
                "Williamson-Hall Analysis Results:",
                f"Slope (4ε): {self.wh_slope:.4e}",
                f"Intercept (Kλ/D): {self.wh_intercept:.4e}"
            ]

            if not np.isnan(size_wh_nm) and size_wh_nm > 0:
                results_lines.append(f"Crystallite Size (D): {size_wh_nm:.2f} nm")
            else:
                results_lines.append("Crystallite Size (D): Could not be reliably calculated (intercept near zero or negative).")

            results_lines.append(f"Strain (ε): {strain_wh:.4e}")

            self._set_results_text(self.wh_results_text_widget, "\n".join(results_lines))
            self.log_message("Williamson-Hall analysis complete.")

            # --- Plotting Williamson-Hall ---
            self.ax_sample.clear() # Clear sample plot to show WH plot
            self.ax_std.clear() # Clear standard plot as well

            self.ax_sample.set_xlabel(r'$4 \sin(\theta)$')
            self.ax_sample.set_ylabel(r'$\beta_{hkl} \cos(\theta)$ (rad)')
            self.ax_sample.set_title('Williamson-Hall Plot')
            self.ax_sample.grid(True)

            self.wh_plot_line, = self.ax_sample.plot(self.wh_x, self.wh_y, 'o', color='blue', label='WH Data Points')

            x_fit_line = np.array([self.wh_x.min(), self.wh_x.max()])
            y_fit_line = self.wh_slope * x_fit_line + self.wh_intercept
            self.wh_fit_line, = self.ax_sample.plot(x_fit_line, y_fit_line, '-', color='red', label='WH Linear Fit')

            self.ax_sample.legend(loc='upper left', fontsize=10)
            self.canvas.draw_idle()

        except Exception as e:
            self._set_results_text(self.wh_results_text_widget, f"Error during Williamson-Hall linear fitting: {e}")
            self.log_message(f"Error during Williamson-Hall linear fitting: {e}")
            self.log_message(f"    Traceback: {traceback.format_exc().strip().splitlines()[-1]}")
            self.update_plot() # Revert plot to standard view if error


if __name__ == "__main__":
    root = tk.Tk()
    app = VolvaCrystalliteAnalysisApp(root)
    root.mainloop()